# Video Transcription with Whisper and WhisperX

In this notebook, we will walk through the process of **transcribing a video** and **generating subtitles** using **OpenAI's Whisper** and **WhisperX** — powerful tools for automatic speech recognition (ASR) and speaker diarization.

**Note:** This notebook may take a significant amount of time to run initially, as it needs to download the required models from the source. Additionally, tasks such as transcription and speaker diarization are computationally intensive. As a reference, processing a 2-minute video can take approximately 5 minutes, even on a high-performance GPU.


# Pre-requisites

To support features of this notebook with CoreAI, we need to install some libraries that are not pre-installed but are required for this notebook. 

## Create and Activate the Virtual Environment:
Open your terminal or command prompt within the Jupyter notebook. Navigate via `File -> New -> Terminal`.
Type `bash` to access a shell compatible with the following commands.
Navigate to the project directory where you want to set up the environment (where this notebook is located):

```bash
export PROJECT_NAME="Video_Transcribing"
export PIP_CACHE_DIR=`pwd`/.cache/pip
mkdir -p $PIP_CACHE_DIR
python -m venv --system-site-packages myvenv
source myvenv/bin/activate
pip install ipykernel
python -m ipykernel install --user --name=${PROJECT_NAME}_myvenv --display-name="Python (${PROJECT_NAME}_myvenv)"
echo ""; echo "Before continuing load the created Python kernel: Python (${PROJECT_NAME}_myvenv)"
```

Load the Python kernel described above before running the cell below (it might take a few seconds for the kernel to appear in the list of kernels).

The following will set the folder location for download so that they are local to the running container, to provide cache.

In [ ]:
import os
os.environ["ANONYMIZED_TELEMETRY"] = 'False'
def set_env_with_cache_dir(env_var_name: str, subdir: str):
    base_cache = os.path.join(os.getcwd(), ".cache")
    full_path = os.path.join(base_cache, subdir)
    os.environ[env_var_name] = full_path
    os.makedirs(full_path, exist_ok=True)
    print(f"{env_var_name}={full_path}")

set_env_with_cache_dir("PIP_CACHE_DIR", "pip")
set_env_with_cache_dir("HF_HOME", "huggingface")
set_env_with_cache_dir("XDG_CACHE_HOME", "whisper")
set_env_with_cache_dir("TORCH_HOME", "torch")
set_env_with_cache_dir("PYANNOTE_CACHE", "pyannote")

## Install Required Libraries:

The rest of this notebook relies on the proper kernel to be loaded and environment variables to be set. 

In [ ]:
!. ./myvenv/bin/activate; pip install -r requirements.txt

## Upload the video
Please upload your video in this folder i.e. `/iti/CoreAI-DemoProjects/Video_Transcribing`

## Setting the paths
The code in the below cell searches for video files in the current working directory and prepares file paths for output files (audio and subtitles). It assumes there is exactly one video file present with a common video extension.

In [ ]:
current_dir = os.getcwd()
files = os.listdir(current_dir)
video_extensions = ['.mp4', '.mkv', '.avi', '.mov', '.flv', '.wmv', '.webm']
video_files = [f for f in files if os.path.splitext(f)[1].lower() in video_extensions]

if len(video_files) == 1:
    video_file = video_files[0]
    video_path = os.path.join(current_dir, video_file)
    
    base_name = os.path.splitext(video_file)[0]
    audio_path = os.path.join(current_dir, f"{base_name}.mp3")
    vtt_file_path= os.path.join(current_dir, f"{base_name}.vtt")
    
    print(f"Video Path: {video_path}")
    print(f"Audio Path: {audio_path}")
    print(f"Subtitles Path: {vtt_file_path}")
else:
    raise FileNotFoundError("There should be exactly one video file in the directory.")

### Extracting Audio from Video

In this step, we use `moviepy`'s `ffmpeg_extract_audio` function to extract the audio track from the video file and save it as an MP3 file. This extracted audio can later be used for tasks like speech-to-text processing.


In [ ]:
from moviepy.video.io.ffmpeg_tools import ffmpeg_extract_audio
ffmpeg_extract_audio(video_path, audio_path)
print("audio saved at:", audio_path)

For downloading the whisperX model you must have a hugging face token (HF_TOKEN) with 'READ' permission

Follow the below steps to generate and use a Hugging Face access token in your local environment or notebook.

### Step 1: Log in to Hugging Face

Go to [Hugging Face Login](https://huggingface.co/login ) and log in with your account.  
If you don't have an account, create one at: [Join Hugging Face](https://huggingface.co/join )

### Step 2: Go to Access Tokens

After logging in:
1. Click on your profile icon (top-right)
2. Select **Settings**
3. Click on **Access Tokens** in the left sidebar

### Step 3: Generate a New Token

1. Click **New token**
2. Give it a name (e.g., `jupyter-token`)
3. Choose the type as `READ`
4. Click **Generate**

**Copy the token now** — you won’t be able to see it again!

### Step 4: Use the Token in Your Notebook

fill the `HF_TOKEN` in the input of the below cell

In [ ]:
HF_TOKEN = input("Enter your HF Token")

In [ ]:
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

### Transcribing Audio Using Whisper

In this step, we use the **Whisper** speech recognition model to transcribe the extracted audio into text. Whisper is a powerful multilingual speech-to-text model developed by OpenAI.

We are using the `large` version of the model in this example, but Whisper offers several sizes with trade-offs between accuracy and speed

The transcription result will include the full text as well as time-stamped segments that can be used to generate subtitles.

In [ ]:
import time
st=time.time()
import whisper
model_name = "large"  
model = whisper.load_model(model_name, DEVICE)
script = model.transcribe(audio_path)
print(f"time taken:{time.time()-st}")

### Performing Speaker Diarization with WhisperX

This step uses **WhisperX**, an extension of OpenAI’s Whisper model, that enables **speaker diarization** — identifying *who spoke when*. This is particularly useful for multi-speaker audio such as interviews, meetings, or lectures.

We use the `DiarizationPipeline` from `whisperx.diarize`, which requires a Hugging Face authentication token (`HF_TOKEN`) to access gated pre-trained models.

### Required Model Access

WhisperX relies on **speaker diarization** and **voice segmentation** models from [`pyannote-audio`](https://github.com/pyannote/pyannote-audio), hosted on Hugging Face. These models are **gated**, meaning you need to manually request access:

1. [pyannote/speaker-diarization-3.1](https://huggingface.co/pyannote/speaker-diarization-3.1)  
2. [pyannote/segmentation-3.0](https://huggingface.co/pyannote/segmentation-3.0)

> **Access Instructions:**  
> - Log in to your Hugging Face account.  
> - Visit the links above.  
> - Fill out the brief access request form.  

Once access is granted, we pass the Hugging Face token directly to WhisperX, which enables the use of the gated models without requiring a separate login step.

In [ ]:
st=time.time()
from whisperx.diarize import DiarizationPipeline
diarization_pipeline = DiarizationPipeline(use_auth_token=HF_TOKEN)
diarized = diarization_pipeline(audio_path)
print(f"time taken:{time.time()-st}")

### Aligning Transcription with Speaker Diarization

Now that we have both the transcription and speaker diarization results, we need to **align them** to produce a final output where each piece of text is assigned to the correct speaker.

#### What We're Doing:
1. **Load an alignment model** based on the detected language.
2. **Align the Whisper transcription** with the actual audio to get more accurate timestamps.
3. **Assign speaker labels** to each word or segment using the diarization result.

This step uses utilities from `whisperx` to perform precise alignment and speaker assignment — critical for generating accurate, speaker-labeled subtitles.

The final `transcribed` list contains time-stamped segments with associated speaker labels — perfect for creating subtitles or transcripts with speaker attribution.

In [ ]:
st=time.time()
from whisperx import load_align_model, align
from whisperx.diarize import assign_word_speakers

model_a, metadata = load_align_model(language_code=script["language"], device=DEVICE)
script_aligned = align(script["segments"], model_a, metadata, audio_path, DEVICE)

result_segments, word_seg = list(assign_word_speakers( diarized, script_aligned).values())
print(f"time taken:{time.time()-st}")

In [ ]:
transcribed = []
for result_segment in result_segments:
    if 'speaker' not in result_segment:
        result_segment['speaker'] = 'unknown'    
    transcribed.append({ "start": result_segment["start"], "end": result_segment["end"], "text": result_segment["text"], "speaker": result_segment["speaker"],})

### Saving Transcribed Output as a `.vtt` Subtitle File

In this final step, we write the speaker-labeled transcription into a **WebVTT** (`.vtt`) file format. This is a standard format used for subtitles in web applications and video players.

Each segment includes:
- A sequence number
- Start and end timestamps (in `HH:MM:SS.mmm` format)
- The transcribed text with speaker attribution

The resulting `.vtt` file can be used directly in video players like HTML5 `<video>` or imported into captioning tools.

In [ ]:
with open(vtt_file_path, 'w') as vtt_file:
    vtt_file.write("WEBVTT\n\n")
    count = 1
    for entry in transcribed:
        start_time = entry["start"]
        end_time = entry["end"]
        text = entry["text"]

        def format_time(seconds):
            hours = int(seconds // 3600)
            minutes = int((seconds % 3600) // 60)
            secs = seconds % 60
            return f"{hours:02}:{minutes:02}:{secs:06.3f}"

        start_time_vtt = format_time(start_time)
        end_time_vtt = format_time(end_time)
        vtt_file.write(f"{count}\n")
        vtt_file.write(f"{start_time_vtt} --> {end_time_vtt}\n")
        vtt_file.write(f"{text}\n\n")
        count += 1  

print(f".vtt file '{vtt_file_path}' created successfully.")

Now we have a fully generated subtitle file with speaker identification! This can be used for captioning videos, accessibility, or content analysis.

In [ ]:
from IPython.display import HTML
video_file = os.path.basename(video_path)
vtt_file = os.path.basename(vtt_file_path)
html_code = f"""
<video width="640" height="360" controls>
  <source src="{video_file}" type="video/webm">
  <track kind="subtitles" src="{vtt_file}" srclang="en" label="English" default>
  Your browser does not support the video tag.
</video>
"""

HTML(html_code)